# Computational Notebook 11: Tokenomics

## Overview

Tokenomics -- the economics of token design -- is the discipline that governs how crypto tokens are created, distributed, and valued within a protocol ecosystem. A well-designed token model aligns the incentives of developers, investors, validators, and users so that the network grows sustainably over time. A poorly designed one can lead to death spirals, governance capture, or runaway inflation. This notebook explores the core building blocks of tokenomics from first principles: supply models (fixed, inflationary, deflationary), distribution and vesting schedules, the equation of exchange applied to crypto, bonding curves for continuous fundraising, fee distribution mechanisms, governance token design, and end-to-end launch simulation. Every model is implemented in pure Python with NumPy so you can experiment with parameters and see how design choices ripple through a token economy.

## Prerequisites
- **Notebook 04**: Smart Contract Development (contract mechanics, token standards)
- **Notebook 05**: DeFi Protocols (AMMs, liquidity pools, yield farming)
- **Notebook 08**: Valuation Models (discounted cash flows, relative valuation)
- **Notebook 10**: Cryptoeconomic Modeling (game theory, mechanism design)
- Basic Python programming and familiarity with NumPy

## Learning Objectives

1. Implement and compare fixed-supply, inflationary, and deflationary token supply models
2. Model token distribution allocations and simulate linear and cliff vesting schedules
3. Apply the equation of exchange (MV = PQ) to estimate token velocity and implied token value
4. Build linear, polynomial, and sigmoid bonding curves and simulate continuous buy/sell dynamics
5. Analyze fee distribution mechanisms including fee-burn (EIP-1559) and staking reward models
6. Simulate vote-escrowed (ve) governance token mechanics and lockup-weighted voting power

## Estimated Time: 4-6 hours

## Related Materials
- See [Section 04: Blockchain Economics](../sections/04-blockchain-economics.md) for foundational economic concepts
- See [Notebook 10: Cryptoeconomic Modeling](./10-cryptoeconomic-modeling.ipynb) for game theory and mechanism design

---

In [ ]:
# ============================================================
# Setup: Import all required libraries
# ============================================================

import sys

def install_and_import(package: str, import_name: str = None) -> None:
    """Try to import a package; install via pip if missing."""
    name = import_name or package
    try:
        __import__(name)
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

install_and_import('numpy')
install_and_import('matplotlib')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Callable
import warnings
warnings.filterwarnings('ignore')

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12

# Reproducibility
np.random.seed(42)

print('All libraries loaded successfully.')
print(f'NumPy: {np.__version__}')

---

# Part 1: Token Supply Models

The supply schedule is the most fundamental design decision in tokenomics. It determines how many tokens exist now, how many will exist in the future, and what economic pressures that creates on price.

We will implement three canonical models:
- **Fixed supply** (Bitcoin-style): A hard cap with diminishing issuance
- **Inflationary** (Ethereum post-merge style): Perpetual issuance at a controlled rate
- **Deflationary** (Burn mechanism): Supply that decreases over time through token burns

In [ ]:
def fixed_supply_model(
    years: int = 30,
    initial_reward: float = 50.0,
    halving_interval: int = 4,
    blocks_per_year: int = 52_560,
    max_supply: float = 21_000_000.0
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Simulate a Bitcoin-style fixed supply model with halvings.

    Args:
        years: Number of years to simulate.
        initial_reward: Block reward at genesis.
        halving_interval: Years between reward halvings.
        blocks_per_year: Approximate blocks mined per year.
        max_supply: Hard cap on total supply.

    Returns:
        Tuple of (time_years, cumulative_supply, annual_issuance).
    """
    time_years = np.arange(0, years + 1, dtype=float)
    cumulative = np.zeros(len(time_years))
    annual_issuance = np.zeros(len(time_years))

    total = 0.0
    for i, year in enumerate(time_years):
        halvings = int(year // halving_interval)
        reward = initial_reward / (2 ** halvings)
        issued = reward * blocks_per_year
        total = min(total + issued, max_supply)
        cumulative[i] = total
        annual_issuance[i] = issued if total < max_supply else 0.0

    return time_years, cumulative, annual_issuance


years_btc, supply_btc, issuance_btc = fixed_supply_model()
print(f'Fixed Supply Model (Bitcoin-style)')
print(f'  Year  5 supply: {supply_btc[5]:>14,.0f} tokens')
print(f'  Year 10 supply: {supply_btc[10]:>14,.0f} tokens')
print(f'  Year 20 supply: {supply_btc[20]:>14,.0f} tokens')
print(f'  Year 30 supply: {supply_btc[30]:>14,.0f} tokens')
print(f'  Hard cap:       {21_000_000:>14,} tokens')

In [ ]:
def inflationary_supply_model(
    years: int = 30,
    initial_supply: float = 100_000_000.0,
    annual_inflation_rate: float = 0.015,
    annual_burn_rate: float = 0.005
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Simulate an Ethereum post-merge style inflationary model.

    Net issuance = staking rewards - base fee burns.

    Args:
        years: Number of years to simulate.
        initial_supply: Starting token supply.
        annual_inflation_rate: Gross inflation from staking rewards.
        annual_burn_rate: Fraction of supply burned annually (EIP-1559).

    Returns:
        Tuple of (time_years, cumulative_supply, net_annual_issuance).
    """
    time_years = np.arange(0, years + 1, dtype=float)
    cumulative = np.zeros(len(time_years))
    net_issuance = np.zeros(len(time_years))

    supply = initial_supply
    for i, _ in enumerate(time_years):
        cumulative[i] = supply
        issued = supply * annual_inflation_rate
        burned = supply * annual_burn_rate
        net = issued - burned
        net_issuance[i] = net
        supply += net

    return time_years, cumulative, net_issuance


years_eth, supply_eth, issuance_eth = inflationary_supply_model()
print(f'Inflationary Model (Ethereum post-merge style)')
print(f'  Net annual inflation: {0.015 - 0.005:.1%}')
print(f'  Year  5 supply: {supply_eth[5]:>14,.0f} tokens')
print(f'  Year 10 supply: {supply_eth[10]:>14,.0f} tokens')
print(f'  Year 30 supply: {supply_eth[30]:>14,.0f} tokens')

In [ ]:
def deflationary_supply_model(
    years: int = 30,
    initial_supply: float = 1_000_000_000.0,
    annual_burn_rate: float = 0.03
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Simulate a deflationary token with a fixed annual burn rate.

    Args:
        years: Number of years to simulate.
        initial_supply: Starting token supply.
        annual_burn_rate: Fraction of supply burned each year.

    Returns:
        Tuple of (time_years, cumulative_supply, annual_burn_amount).
    """
    time_years = np.arange(0, years + 1, dtype=float)
    cumulative = np.zeros(len(time_years))
    annual_burn = np.zeros(len(time_years))

    supply = initial_supply
    for i in range(len(time_years)):
        cumulative[i] = supply
        burn = supply * annual_burn_rate
        annual_burn[i] = burn
        supply -= burn

    return time_years, cumulative, annual_burn


years_def, supply_def, burn_def = deflationary_supply_model()
print(f'Deflationary Model (Burn mechanism)')
print(f'  Annual burn rate: {0.03:.1%}')
print(f'  Year  5 supply: {supply_def[5]:>14,.0f} tokens')
print(f'  Year 10 supply: {supply_def[10]:>14,.0f} tokens')
print(f'  Year 30 supply: {supply_def[30]:>14,.0f} tokens')
print(f'  Supply reduction by year 30: {(1 - supply_def[30]/supply_def[0]):.1%}')

In [ ]:
# Visualize all three supply models side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Fixed supply
axes[0].plot(years_btc, supply_btc / 1e6, color='#F7931A', linewidth=2)
axes[0].axhline(y=21, color='gray', linestyle='--', alpha=0.7, label='21M cap')
axes[0].set_title('Fixed Supply (Bitcoin-style)', fontsize=13)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Supply (millions)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Inflationary
axes[1].plot(years_eth, supply_eth / 1e6, color='#627EEA', linewidth=2)
axes[1].set_title('Inflationary (ETH post-merge style)', fontsize=13)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Supply (millions)')
axes[1].grid(True, alpha=0.3)

# Deflationary
axes[2].plot(years_def, supply_def / 1e6, color='#E74C3C', linewidth=2)
axes[2].set_title('Deflationary (Burn mechanism)', fontsize=13)
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Supply (millions)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Token Supply Models Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Figure: Three canonical supply models showing distinct long-term trajectories.')

### Key Takeaways -- Supply Models

| Model | Scarcity Signal | Inflation Risk | Use Case |
|-------|----------------|---------------|----------|
| **Fixed** | Strong -- provable cap | None once cap reached | Store-of-value assets (BTC) |
| **Inflationary** | Moderate -- depends on net issuance | Controlled if burns offset | Staking networks (ETH) |
| **Deflationary** | Increasing over time | None -- supply only shrinks | Fee-burn tokens (BNB) |

The choice of supply model directly impacts holder psychology, validator incentives, and long-term price dynamics.

---

# Part 2: Token Distribution & Vesting

How tokens are allocated at launch -- and how those allocations unlock over time -- determines the initial ownership structure and the sell pressure the market faces in the months and years after launch.

Typical allocation categories:
- **Team / Founders**: Core contributors (usually 15-20%)
- **Investors**: Seed, private, and public sale participants (10-25%)
- **Community / Ecosystem**: Grants, airdrops, liquidity mining (30-50%)
- **Treasury / Reserve**: Protocol-controlled funds (10-20%)

Vesting schedules prevent early holders from dumping tokens immediately. The two most common patterns are **linear vesting** (steady unlock over time) and **cliff vesting** (nothing unlocks until a cliff date, then linear thereafter).

In [ ]:
@dataclass
class Allocation:
    """Represents a token allocation bucket with its vesting schedule."""
    name: str
    total_tokens: float
    tge_unlock_pct: float   # % unlocked at Token Generation Event
    cliff_months: int       # months before vesting begins
    vesting_months: int     # linear vesting duration after cliff
    color: str = '#333333'


def compute_unlock_schedule(
    allocation: Allocation,
    months: int = 48
) -> np.ndarray:
    """Compute monthly cumulative unlocked tokens for an allocation.

    Args:
        allocation: The allocation with vesting parameters.
        months: Total months to simulate.

    Returns:
        Array of cumulative unlocked tokens at each month.
    """
    timeline = np.zeros(months + 1)
    total = allocation.total_tokens
    tge_amount = total * allocation.tge_unlock_pct
    vesting_amount = total - tge_amount

    for m in range(months + 1):
        unlocked = tge_amount
        if m > allocation.cliff_months and allocation.vesting_months > 0:
            vested_months = min(m - allocation.cliff_months, allocation.vesting_months)
            unlocked += vesting_amount * (vested_months / allocation.vesting_months)
        elif m > allocation.cliff_months and allocation.vesting_months == 0:
            unlocked = total
        timeline[m] = unlocked

    return timeline


# Define a realistic token launch allocation
TOTAL_SUPPLY = 1_000_000_000  # 1 billion tokens

allocations = [
    Allocation('Team',      TOTAL_SUPPLY * 0.18, tge_unlock_pct=0.0,  cliff_months=12, vesting_months=36, color='#E74C3C'),
    Allocation('Investors', TOTAL_SUPPLY * 0.15, tge_unlock_pct=0.0,  cliff_months=6,  vesting_months=24, color='#3498DB'),
    Allocation('Community', TOTAL_SUPPLY * 0.40, tge_unlock_pct=0.10, cliff_months=0,  vesting_months=48, color='#2ECC71'),
    Allocation('Treasury',  TOTAL_SUPPLY * 0.20, tge_unlock_pct=0.0,  cliff_months=6,  vesting_months=36, color='#F39C12'),
    Allocation('Public Sale', TOTAL_SUPPLY * 0.07, tge_unlock_pct=0.50, cliff_months=0, vesting_months=6, color='#9B59B6'),
]

print('Token Distribution Plan')
print('=' * 50)
for a in allocations:
    print(f'  {a.name:<12} {a.total_tokens/1e6:>8.1f}M  ({a.total_tokens/TOTAL_SUPPLY:>5.1%})  '
          f'TGE: {a.tge_unlock_pct:>4.0%}  Cliff: {a.cliff_months:>2}mo  Vest: {a.vesting_months:>2}mo')
print(f'  {"TOTAL":<12} {TOTAL_SUPPLY/1e6:>8.1f}M')

In [ ]:
# Compute and visualize unlock timelines
SIM_MONTHS = 48
months_axis = np.arange(0, SIM_MONTHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Stacked area: circulating supply over time
schedules = []
for alloc in allocations:
    sched = compute_unlock_schedule(alloc, SIM_MONTHS)
    schedules.append(sched)

stacked = np.row_stack(schedules)
ax1.stackplot(
    months_axis, stacked / 1e6,
    labels=[a.name for a in allocations],
    colors=[a.color for a in allocations],
    alpha=0.85
)
ax1.set_title('Circulating Supply Unlock Over Time', fontsize=13)
ax1.set_xlabel('Month')
ax1.set_ylabel('Unlocked Tokens (millions)')
ax1.legend(loc='upper left', fontsize=10)
ax1.grid(True, alpha=0.3)

# Per-category unlock curves
for alloc, sched in zip(allocations, schedules):
    ax2.plot(months_axis, sched / alloc.total_tokens * 100,
             label=alloc.name, color=alloc.color, linewidth=2)
ax2.set_title('Vesting Progress by Category', fontsize=13)
ax2.set_xlabel('Month')
ax2.set_ylabel('% Unlocked')
ax2.legend(fontsize=10)
ax2.set_ylim(0, 105)
ax2.grid(True, alpha=0.3)

plt.suptitle('Token Distribution & Vesting Schedules', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

total_unlocked = np.sum(stacked, axis=0)
print(f'Circulating supply at TGE (month 0): {total_unlocked[0]/1e6:>8.1f}M ({total_unlocked[0]/TOTAL_SUPPLY:.1%})')
print(f'Circulating supply at month 12:      {total_unlocked[12]/1e6:>8.1f}M ({total_unlocked[12]/TOTAL_SUPPLY:.1%})')
print(f'Circulating supply at month 24:      {total_unlocked[24]/1e6:>8.1f}M ({total_unlocked[24]/TOTAL_SUPPLY:.1%})')
print(f'Circulating supply at month 48:      {total_unlocked[48]/1e6:>8.1f}M ({total_unlocked[48]/TOTAL_SUPPLY:.1%})')

### Impact of Vesting on Sell Pressure

The monthly *marginal* unlock -- how many new tokens become liquid each month -- is a proxy for potential sell pressure. Large cliff unlocks create sharp spikes that can crater the price.

In [ ]:
# Monthly marginal unlock (new tokens entering circulation each month)
marginal_unlock = np.diff(total_unlocked)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(months_axis[1:], marginal_unlock / 1e6, color='#E74C3C', alpha=0.7, width=0.8)
ax.set_title('Monthly New Token Unlocks (Sell Pressure Proxy)', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Newly Unlocked Tokens (millions)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

peak_month = int(np.argmax(marginal_unlock)) + 1
print(f'Peak unlock month: {peak_month} ({marginal_unlock[peak_month-1]/1e6:.1f}M tokens)')
print(f'Average monthly unlock: {marginal_unlock.mean()/1e6:.1f}M tokens')

---

# Part 3: Token Velocity & The Equation of Exchange

The **Equation of Exchange** from monetary economics applies directly to token valuation:

$$MV = PQ$$

Where:
- **M** = Monetary base (market cap / total value of tokens)
- **V** = Velocity (how many times each token changes hands per period)
- **P** = Price level (price of goods/services in the ecosystem)
- **Q** = Quantity of goods/services transacted

Rearranging: **Token Price = PQ / (Supply x V)**

High velocity means tokens are spent quickly (like a medium of exchange), which *reduces* the value each token must carry. Staking, lockups, and governance participation reduce velocity, which *increases* implied token value.

In [ ]:
def equation_of_exchange(
    gdp: float,
    velocity: float,
    circulating_supply: float
) -> float:
    """Calculate implied token price from the equation of exchange.

    MV = PQ, so Token Price = PQ / (Supply * V) = GDP / (Supply * V).

    Args:
        gdp: Total economic output of the network (PQ).
        velocity: Token velocity (transactions per token per period).
        circulating_supply: Number of tokens in circulation.

    Returns:
        Implied price per token.
    """
    if velocity <= 0 or circulating_supply <= 0:
        return 0.0
    return gdp / (circulating_supply * velocity)


# Example: a DeFi protocol with known on-chain GDP
network_gdp = 5_000_000_000   # $5B annual transaction volume
supply = 500_000_000          # 500M tokens circulating

velocities = [2, 5, 10, 20, 50]
print('Equation of Exchange: Implied Token Prices')
print('=' * 50)
print(f'  Network GDP:        ${network_gdp/1e9:.1f}B')
print(f'  Circulating Supply: {supply/1e6:.0f}M tokens')
print()
for v in velocities:
    price = equation_of_exchange(network_gdp, v, supply)
    mcap = price * supply
    print(f'  Velocity = {v:>2}x  =>  Price = ${price:>8.2f}  |  Market Cap = ${mcap/1e9:.2f}B')

In [ ]:
def model_velocity_with_staking(
    base_velocity: float,
    staking_pcts: np.ndarray
) -> np.ndarray:
    """Model effective velocity as staking reduces circulating float.

    Effective velocity increases on remaining float, but total network
    velocity decreases because staked tokens have velocity = 0.

    Effective V = base_velocity * (1 - staking_pct).

    Args:
        base_velocity: Velocity when no tokens are staked.
        staking_pcts: Array of staking percentages to evaluate.

    Returns:
        Array of effective velocities.
    """
    return base_velocity * (1.0 - staking_pcts)


staking_range = np.linspace(0, 0.9, 100)
base_v = 10.0
effective_v = model_velocity_with_staking(base_v, staking_range)
implied_prices = np.array([
    equation_of_exchange(network_gdp, v, supply) for v in effective_v
])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(staking_range * 100, effective_v, color='#3498DB', linewidth=2)
ax1.set_title('Effective Velocity vs Staking Rate', fontsize=13)
ax1.set_xlabel('% of Supply Staked')
ax1.set_ylabel('Effective Velocity')
ax1.grid(True, alpha=0.3)

ax2.plot(staking_range * 100, implied_prices, color='#2ECC71', linewidth=2)
ax2.set_title('Implied Token Price vs Staking Rate', fontsize=13)
ax2.set_xlabel('% of Supply Staked')
ax2.set_ylabel('Implied Price ($)')
ax2.grid(True, alpha=0.3)

plt.suptitle('Staking Reduces Velocity and Increases Token Value', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'At  0% staked: price = ${implied_prices[0]:.2f}')
print(f'At 30% staked: price = ${implied_prices[33]:.2f}')
print(f'At 60% staked: price = ${implied_prices[66]:.2f}')
print(f'At 90% staked: price = ${implied_prices[99]:.2f}')

### Token Velocity Sinks

Protocols use several mechanisms to reduce velocity and increase token value:

1. **Staking**: Lock tokens to earn rewards and secure the network
2. **Governance locking**: Lock tokens for voting power (veToken model)
3. **Collateral**: Lock tokens as collateral for borrowing or minting
4. **Buy-and-burn**: Reduce supply permanently using protocol fees
5. **LP provision**: Tokens locked in liquidity pools

Each of these velocity sinks has different trade-offs between capital efficiency and value accrual.

---

# Part 4: Bonding Curves

A **bonding curve** is a mathematical function that defines the relationship between a token's price and its supply. When someone buys tokens, new tokens are minted along the curve and the price increases. When someone sells, tokens are burned and the price decreases. This creates a deterministic, automated market with continuous liquidity.

Key properties:
- Price is a function of supply: $P = f(S)$
- Total reserve (collateral locked) is the integral: $R = \int_0^S f(s) \, ds$
- Buys move right along the curve (price up), sells move left (price down)

In [ ]:
def linear_bonding_curve(supply: np.ndarray, slope: float = 0.001, intercept: float = 0.1) -> np.ndarray:
    """Linear bonding curve: P = slope * S + intercept.

    Args:
        supply: Array of supply values.
        slope: Price increase per token.
        intercept: Starting price at zero supply.

    Returns:
        Array of prices.
    """
    return slope * supply + intercept


def polynomial_bonding_curve(supply: np.ndarray, coefficient: float = 1e-8, power: float = 2.0, intercept: float = 0.01) -> np.ndarray:
    """Polynomial bonding curve: P = coefficient * S^power + intercept.

    Args:
        supply: Array of supply values.
        coefficient: Scaling coefficient.
        power: Exponent (2 = quadratic, 3 = cubic, etc.).
        intercept: Starting price at zero supply.

    Returns:
        Array of prices.
    """
    return coefficient * np.power(supply, power) + intercept


def sigmoid_bonding_curve(supply: np.ndarray, max_price: float = 10.0, midpoint: float = 5000.0, steepness: float = 0.002) -> np.ndarray:
    """Sigmoid bonding curve: P = max_price / (1 + exp(-steepness * (S - midpoint))).

    Args:
        supply: Array of supply values.
        max_price: Asymptotic maximum price.
        midpoint: Supply at which price = max_price / 2.
        steepness: Controls how quickly price transitions.

    Returns:
        Array of prices.
    """
    return max_price / (1.0 + np.exp(-steepness * (supply - midpoint)))


supply_range = np.linspace(0, 10000, 500)

p_linear = linear_bonding_curve(supply_range)
p_poly = polynomial_bonding_curve(supply_range)
p_sigmoid = sigmoid_bonding_curve(supply_range)

print('Bonding Curve Prices at Key Supply Levels')
print('=' * 60)
for s in [0, 1000, 5000, 10000]:
    idx = int(s / 10000 * 499)
    print(f'  Supply = {s:>6}  |  Linear: ${p_linear[idx]:.4f}  '
          f'Polynomial: ${p_poly[idx]:.4f}  Sigmoid: ${p_sigmoid[idx]:.4f}')

In [ ]:
# Visualize the three bonding curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(supply_range, p_linear, color='#3498DB', linewidth=2)
axes[0].fill_between(supply_range, p_linear, alpha=0.15, color='#3498DB')
axes[0].set_title('Linear: P = mS + b', fontsize=13)
axes[0].set_xlabel('Token Supply')
axes[0].set_ylabel('Price ($)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(supply_range, p_poly, color='#E74C3C', linewidth=2)
axes[1].fill_between(supply_range, p_poly, alpha=0.15, color='#E74C3C')
axes[1].set_title('Polynomial: P = aS² + b', fontsize=13)
axes[1].set_xlabel('Token Supply')
axes[1].set_ylabel('Price ($)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(supply_range, p_sigmoid, color='#2ECC71', linewidth=2)
axes[2].fill_between(supply_range, p_sigmoid, alpha=0.15, color='#2ECC71')
axes[2].set_title('Sigmoid: P = M / (1 + e^(-k(S-m)))', fontsize=13)
axes[2].set_xlabel('Token Supply')
axes[2].set_ylabel('Price ($)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Bonding Curve Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Shaded area under each curve represents the total reserve (collateral locked).')

In [ ]:
@dataclass
class BondingCurveMarket:
    """Simulates a bonding curve token market with buy/sell operations."""
    price_function: Callable[[np.ndarray], np.ndarray]
    current_supply: float = 0.0
    reserve_balance: float = 0.0
    history: List[Dict] = field(default_factory=list)

    def buy(self, amount: float) -> float:
        """Buy `amount` tokens. Returns the cost in reserve currency.

        Args:
            amount: Number of tokens to purchase.

        Returns:
            Total cost paid.
        """
        # Numerical integration: cost = integral of price from S to S + amount
        steps = 100
        s_vals = np.linspace(self.current_supply, self.current_supply + amount, steps)
        prices = self.price_function(s_vals)
        cost = float(np.trapz(prices, s_vals))

        self.current_supply += amount
        self.reserve_balance += cost
        self.history.append({
            'action': 'buy', 'amount': amount, 'cost': cost,
            'price': float(self.price_function(np.array([self.current_supply]))[0]),
            'supply': self.current_supply, 'reserve': self.reserve_balance
        })
        return cost

    def sell(self, amount: float) -> float:
        """Sell `amount` tokens. Returns the payout from the reserve.

        Args:
            amount: Number of tokens to sell.

        Returns:
            Total payout received.
        """
        amount = min(amount, self.current_supply)
        steps = 100
        s_vals = np.linspace(self.current_supply - amount, self.current_supply, steps)
        prices = self.price_function(s_vals)
        payout = float(np.trapz(prices, s_vals))

        self.current_supply -= amount
        self.reserve_balance -= payout
        self.history.append({
            'action': 'sell', 'amount': amount, 'cost': -payout,
            'price': float(self.price_function(np.array([self.current_supply]))[0]),
            'supply': self.current_supply, 'reserve': self.reserve_balance
        })
        return payout


# Simulate buy/sell dynamics on a polynomial curve
market = BondingCurveMarket(price_function=polynomial_bonding_curve)

actions = [
    ('buy', 2000), ('buy', 1500), ('buy', 3000),
    ('sell', 1000), ('buy', 500), ('sell', 2000),
    ('buy', 1000), ('sell', 500),
]

print('Bonding Curve Market Simulation (Polynomial)')
print('=' * 75)
print(f'{"Action":<8} {"Amount":>8} {"Cost/Payout":>14} {"Price":>10} {"Supply":>10} {"Reserve":>12}')
print('-' * 75)

for action, amount in actions:
    if action == 'buy':
        cost = market.buy(amount)
    else:
        cost = market.sell(amount)
    h = market.history[-1]
    cost_str = f'${abs(h["cost"]):>11,.2f}'
    print(f'{action.upper():<8} {amount:>8,} {cost_str:>14} ${h["price"]:>8.4f} {h["supply"]:>10,.0f} ${h["reserve"]:>10,.2f}')

In [ ]:
# Visualize the buy/sell simulation
prices_hist = [h['price'] for h in market.history]
supplies_hist = [h['supply'] for h in market.history]
reserves_hist = [h['reserve'] for h in market.history]
colors = ['#2ECC71' if h['action'] == 'buy' else '#E74C3C' for h in market.history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Price trajectory on the bonding curve
curve_supply = np.linspace(0, 8000, 300)
curve_price = polynomial_bonding_curve(curve_supply)
ax1.plot(curve_supply, curve_price, color='gray', linewidth=1, alpha=0.5, label='Bonding Curve')
ax1.scatter(supplies_hist, prices_hist, c=colors, s=100, zorder=5, edgecolors='black')
for i in range(len(supplies_hist) - 1):
    ax1.annotate('', xy=(supplies_hist[i+1], prices_hist[i+1]),
                 xytext=(supplies_hist[i], prices_hist[i]),
                 arrowprops=dict(arrowstyle='->', color=colors[i+1], lw=1.5))
ax1.set_title('Price Movement Along Bonding Curve', fontsize=13)
ax1.set_xlabel('Token Supply')
ax1.set_ylabel('Price ($)')
ax1.grid(True, alpha=0.3)

# Reserve balance over time
ax2.bar(range(len(reserves_hist)), reserves_hist, color=colors, alpha=0.7, edgecolor='black')
ax2.set_title('Reserve Balance After Each Transaction', fontsize=13)
ax2.set_xlabel('Transaction #')
ax2.set_ylabel('Reserve ($)')
ax2.grid(True, alpha=0.3)

plt.suptitle('Bonding Curve Buy/Sell Dynamics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Green = buy (mint), Red = sell (burn)')

### Bonding Curves Enable Continuous Fundraising

Unlike traditional fundraising (fixed-price ICO), bonding curves:
- **Reward early participants** with lower prices
- **Guarantee liquidity** -- you can always sell back to the curve
- **Build a reserve** that backs every token in circulation
- **Eliminate the need for external exchanges** during bootstrapping

The shape of the curve (linear, polynomial, sigmoid) controls the trade-off between early-adopter advantage and price stability.

---

# Part 5: Fee Distribution & Value Accrual

How a protocol distributes its fees determines how value flows to token holders. The two dominant models are:

1. **Fee-burn (EIP-1559 style)**: Protocol fees are used to buy and burn tokens, reducing supply and increasing scarcity.
2. **Fee-distribution (staking rewards)**: Protocol fees are distributed directly to stakers/validators as income.

Each model has different implications for token valuation, tax treatment, and holder behavior.

In [ ]:
def simulate_fee_burn(
    years: int = 10,
    initial_supply: float = 500_000_000.0,
    annual_fees: float = 100_000_000.0,
    initial_token_price: float = 2.0,
    fee_growth_rate: float = 0.10
) -> Dict[str, np.ndarray]:
    """Simulate a fee-burn model where protocol fees buy and burn tokens.

    Args:
        years: Simulation duration.
        initial_supply: Starting token supply.
        annual_fees: Protocol fee revenue in year 1.
        initial_token_price: Starting token price.
        fee_growth_rate: Annual growth in protocol fees.

    Returns:
        Dictionary with time series of supply, price, burned, and fees.
    """
    supply = initial_supply
    price = initial_token_price
    time_arr = np.arange(years + 1)
    supplies = np.zeros(years + 1)
    prices = np.zeros(years + 1)
    burned_arr = np.zeros(years + 1)
    fees_arr = np.zeros(years + 1)

    for y in range(years + 1):
        supplies[y] = supply
        prices[y] = price
        fees = annual_fees * ((1 + fee_growth_rate) ** y)
        fees_arr[y] = fees
        tokens_burned = fees / price
        burned_arr[y] = tokens_burned
        supply -= tokens_burned
        # Simple price model: market cap stays proportional to fees, supply drops
        market_cap = fees * 15  # 15x fee multiple
        price = market_cap / supply if supply > 0 else 0

    return {'time': time_arr, 'supply': supplies, 'price': prices,
            'burned': burned_arr, 'fees': fees_arr}


def simulate_fee_distribution(
    years: int = 10,
    total_supply: float = 500_000_000.0,
    staking_pct: float = 0.40,
    annual_fees: float = 100_000_000.0,
    fee_growth_rate: float = 0.10
) -> Dict[str, np.ndarray]:
    """Simulate fee-distribution model where fees go to stakers.

    Args:
        years: Simulation duration.
        total_supply: Total token supply (constant).
        staking_pct: Fraction of supply staked.
        annual_fees: Protocol fee revenue in year 1.
        fee_growth_rate: Annual growth in protocol fees.

    Returns:
        Dictionary with time series of yield, fees, and cumulative rewards.
    """
    staked_tokens = total_supply * staking_pct
    time_arr = np.arange(years + 1)
    yields = np.zeros(years + 1)
    fees_arr = np.zeros(years + 1)
    cumulative_rewards = np.zeros(years + 1)

    total_rewards = 0.0
    for y in range(years + 1):
        fees = annual_fees * ((1 + fee_growth_rate) ** y)
        fees_arr[y] = fees
        reward_per_token = fees / staked_tokens
        staker_yield = fees / (staked_tokens * 2.0)  # assume $2 token price
        yields[y] = staker_yield
        total_rewards += fees
        cumulative_rewards[y] = total_rewards

    return {'time': time_arr, 'yields': yields, 'fees': fees_arr,
            'cumulative_rewards': cumulative_rewards}


burn_result = simulate_fee_burn()
dist_result = simulate_fee_distribution()

print('Fee-Burn Model Summary (10 years)')
print(f'  Starting supply: {burn_result["supply"][0]/1e6:.0f}M')
print(f'  Ending supply:   {burn_result["supply"][-1]/1e6:.0f}M')
print(f'  Total burned:    {burn_result["burned"].sum()/1e6:.1f}M tokens')
print(f'  Starting price:  ${burn_result["price"][0]:.2f}')
print(f'  Ending price:    ${burn_result["price"][-1]:.2f}')
print()
print('Fee-Distribution Model Summary (10 years)')
print(f'  Staker yield year 0: {dist_result["yields"][0]:.1%}')
print(f'  Staker yield year 10: {dist_result["yields"][-1]:.1%}')
print(f'  Cumulative fees distributed: ${dist_result["cumulative_rewards"][-1]/1e9:.2f}B')

In [ ]:
# Visualize both fee models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Supply comparison
axes[0].plot(burn_result['time'], burn_result['supply'] / 1e6, color='#E74C3C', linewidth=2, label='Fee-Burn')
axes[0].axhline(y=500, color='#3498DB', linestyle='--', linewidth=2, label='Fee-Distribution (constant)')
axes[0].set_title('Token Supply Over Time', fontsize=13)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Supply (millions)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Price trajectory (burn model)
axes[1].plot(burn_result['time'], burn_result['price'], color='#E74C3C', linewidth=2)
axes[1].set_title('Token Price (Fee-Burn Model)', fontsize=13)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Price ($)')
axes[1].grid(True, alpha=0.3)

# Staker yield (distribution model)
axes[2].plot(dist_result['time'], dist_result['yields'] * 100, color='#3498DB', linewidth=2)
axes[2].set_title('Staker Yield (Fee-Distribution)', fontsize=13)
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Annual Yield (%)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Fee-Burn vs Fee-Distribution Models', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Fee-burn increases price through scarcity; fee-distribution provides income to stakers.')

### Value Accrual Comparison

| Mechanism | Value Accrual | Holder Behavior | Tax Implications |
|-----------|--------------|-----------------|------------------|
| **Fee-Burn** | Indirect (price appreciation) | Passive -- just hold | Capital gains only |
| **Fee-Distribution** | Direct (income stream) | Active -- must stake | Income tax + capital gains |
| **Hybrid** | Both channels | Flexible | Depends on mix |

Many modern protocols use a hybrid approach: burn a portion of fees and distribute the rest to stakers.

---

# Part 6: Governance Token Valuation

Governance tokens grant holders the right to vote on protocol parameters, treasury allocation, and upgrades. The **vote-escrowed (ve)** model, pioneered by Curve Finance (veCRV), introduces time-weighted voting: the longer you lock your tokens, the more voting power you receive.

Key mechanics of the veToken model:
- Lock tokens for a duration (e.g., 1 week to 4 years)
- Voting power = locked amount * (remaining lock time / max lock time)
- Voting power decays linearly as the lock approaches expiry
- Longer locks earn higher share of protocol fees and governance influence

In [ ]:
@dataclass
class VeTokenLock:
    """Represents a vote-escrowed token lock position."""
    owner: str
    amount: float
    lock_duration_weeks: int
    lock_start_week: int = 0

    @property
    def lock_end_week(self) -> int:
        return self.lock_start_week + self.lock_duration_weeks

    def voting_power(self, current_week: int, max_lock_weeks: int = 208) -> float:
        """Calculate voting power at a given week.

        Power decays linearly from (amount * duration/max) to 0.

        Args:
            current_week: The current week number.
            max_lock_weeks: Maximum allowed lock duration (4 years = 208 weeks).

        Returns:
            Voting power at the current week.
        """
        if current_week >= self.lock_end_week:
            return 0.0
        remaining = self.lock_end_week - current_week
        return self.amount * (remaining / max_lock_weeks)


# Demonstrate voting power decay
max_lock = 208  # 4 years in weeks
lock_durations = [52, 104, 156, 208]  # 1, 2, 3, 4 years
amount = 1000.0

print('Vote-Escrowed Token Mechanics')
print('=' * 60)
print(f'Locked Amount: {amount:,.0f} tokens')
print(f'Max Lock: {max_lock} weeks (4 years)')
print()
print(f'{"Lock Duration":<15} {"Initial vePower":>15} {"Power at 1yr":>15} {"Multiplier":>12}')
print('-' * 60)
for dur in lock_durations:
    lock = VeTokenLock(owner='Alice', amount=amount, lock_duration_weeks=dur)
    initial = lock.voting_power(0, max_lock)
    at_1yr = lock.voting_power(52, max_lock)
    mult = initial / amount
    print(f'{dur:>3} weeks ({dur//52}yr)   {initial:>14,.1f} {at_1yr:>14,.1f} {mult:>11.2f}x')

In [ ]:
# Visualize voting power decay for different lock durations
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors_ve = ['#3498DB', '#2ECC71', '#F39C12', '#E74C3C']

weeks = np.arange(0, 210)
for dur, color in zip(lock_durations, colors_ve):
    lock = VeTokenLock(owner='User', amount=1000, lock_duration_weeks=dur)
    powers = [lock.voting_power(w, max_lock) for w in weeks]
    ax1.plot(weeks, powers, color=color, linewidth=2, label=f'{dur//52}yr lock')

ax1.set_title('Voting Power Decay Over Time', fontsize=13)
ax1.set_xlabel('Week')
ax1.set_ylabel('Voting Power (veTOKEN)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Fee share based on voting power proportion
# Simulate a pool of lockers with different durations
np.random.seed(42)
n_lockers = 200
locker_amounts = np.random.lognormal(mean=7, sigma=1.5, size=n_lockers)
locker_durations = np.random.choice(lock_durations, size=n_lockers, p=[0.3, 0.3, 0.2, 0.2])

locks = [VeTokenLock(f'User_{i}', amt, dur) for i, (amt, dur) in enumerate(zip(locker_amounts, locker_durations))]

total_power_by_duration = {}
for dur in lock_durations:
    dur_locks = [l for l in locks if l.lock_duration_weeks == dur]
    total_power_by_duration[f'{dur//52}yr'] = sum(l.voting_power(0, max_lock) for l in dur_locks)

labels = list(total_power_by_duration.keys())
values = list(total_power_by_duration.values())
ax2.bar(labels, [v/1e6 for v in values], color=colors_ve, alpha=0.8, edgecolor='black')
ax2.set_title('Total Voting Power by Lock Duration', fontsize=13)
ax2.set_xlabel('Lock Duration')
ax2.set_ylabel('Aggregate Voting Power (millions)')
ax2.grid(True, alpha=0.3)

plt.suptitle('veToken Governance Mechanics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

total_all = sum(values)
print('Fee Share by Lock Duration (at week 0):')
for label, val in zip(labels, values):
    print(f'  {label} lockers: {val/total_all:.1%} of total voting power')

In [ ]:
def simulate_ve_governance_vote(
    locks: List[VeTokenLock],
    current_week: int,
    max_lock_weeks: int = 208
) -> Dict[str, float]:
    """Simulate a governance vote among veToken holders.

    Randomly assigns each locker to vote for option A or B.
    Votes are weighted by voting power.

    Args:
        locks: List of VeTokenLock positions.
        current_week: Week at which the vote takes place.
        max_lock_weeks: Maximum lock duration for power calculation.

    Returns:
        Dictionary with vote tallies and outcome.
    """
    votes_a = 0.0
    votes_b = 0.0
    n_a = 0
    n_b = 0

    for lock in locks:
        power = lock.voting_power(current_week, max_lock_weeks)
        if power <= 0:
            continue
        if np.random.random() < 0.55:  # slight preference for option A
            votes_a += power
            n_a += 1
        else:
            votes_b += power
            n_b += 1

    total = votes_a + votes_b
    return {
        'option_a_power': votes_a,
        'option_b_power': votes_b,
        'option_a_pct': votes_a / total if total > 0 else 0,
        'option_b_pct': votes_b / total if total > 0 else 0,
        'n_voters_a': n_a,
        'n_voters_b': n_b,
        'winner': 'A' if votes_a > votes_b else 'B'
    }


np.random.seed(42)
vote_result = simulate_ve_governance_vote(locks, current_week=0)
print('Governance Vote Simulation (Week 0)')
print('=' * 50)
print(f'  Option A: {vote_result["n_voters_a"]:>3} voters, {vote_result["option_a_power"]/1e6:>8.2f}M vePower ({vote_result["option_a_pct"]:.1%})')
print(f'  Option B: {vote_result["n_voters_b"]:>3} voters, {vote_result["option_b_power"]/1e6:>8.2f}M vePower ({vote_result["option_b_pct"]:.1%})')
print(f'  Winner:   Option {vote_result["winner"]}')
print()
print('Note: A whale with a 4-year lock can outvote many short-term holders.')
print('This is by design -- veTokens reward long-term commitment to the protocol.')

---

# Part 7: Token Launch Simulation

We now combine all the building blocks from Parts 1-6 into a comprehensive token launch simulator. We will:

1. Set initial parameters (supply, allocations, vesting, bonding curve)
2. Simulate 24 months of market activity
3. Track circulating supply, price, velocity, burns, and treasury value
4. Visualize the complete lifecycle

In [ ]:
@dataclass
class TokenLaunchConfig:
    """Configuration for a token launch simulation."""
    name: str
    total_supply: float
    initial_price: float
    allocations: List[Allocation]
    monthly_volume_base: float       # base monthly trading volume in USD
    monthly_volume_growth: float     # monthly growth rate of volume
    fee_rate: float                  # protocol fee as fraction of volume
    burn_pct: float                  # fraction of fees burned
    staking_pct: float              # fraction of circulating supply staked
    base_velocity: float            # token velocity without staking


@dataclass
class LaunchSimResult:
    """Results from a token launch simulation."""
    months: np.ndarray
    circulating_supply: np.ndarray
    total_supply: np.ndarray
    price: np.ndarray
    market_cap: np.ndarray
    volume: np.ndarray
    fees_collected: np.ndarray
    tokens_burned: np.ndarray
    treasury_value: np.ndarray
    velocity: np.ndarray


def simulate_token_launch(
    config: TokenLaunchConfig,
    duration_months: int = 24
) -> LaunchSimResult:
    """Run an end-to-end token launch simulation.

    Combines vesting unlocks, trading volume, fee accrual,
    token burns, staking effects, and velocity to produce
    a month-by-month picture of the token economy.

    Args:
        config: Token launch configuration.
        duration_months: Number of months to simulate.

    Returns:
        LaunchSimResult with all tracked metrics.
    """
    n = duration_months + 1
    months = np.arange(n)

    # Pre-compute vesting schedules
    unlock_schedules = [compute_unlock_schedule(a, duration_months) for a in config.allocations]
    total_unlocked = np.sum(np.row_stack(unlock_schedules), axis=0)

    circ_supply = np.zeros(n)
    total_sup = np.full(n, config.total_supply)
    price = np.zeros(n)
    mcap = np.zeros(n)
    volume = np.zeros(n)
    fees = np.zeros(n)
    burned = np.zeros(n)
    treasury = np.zeros(n)
    velocity = np.zeros(n)

    current_total = config.total_supply
    current_price = config.initial_price
    treasury_balance = 0.0

    for m in range(n):
        # Circulating supply from vesting
        circ = total_unlocked[m]

        # Subtract cumulative burns from total supply
        total_sup[m] = current_total
        circ_supply[m] = min(circ, current_total)

        # Trading volume
        vol = config.monthly_volume_base * ((1 + config.monthly_volume_growth) ** m)
        # Add randomness
        vol *= np.random.uniform(0.7, 1.3)
        volume[m] = vol

        # Fees
        fee = vol * config.fee_rate
        fees[m] = fee

        # Burns
        burn_usd = fee * config.burn_pct
        tokens_to_burn = burn_usd / current_price if current_price > 0 else 0
        burned[m] = tokens_to_burn
        current_total -= tokens_to_burn

        # Treasury gets remaining fees
        treasury_balance += fee * (1 - config.burn_pct)
        treasury[m] = treasury_balance

        # Velocity
        eff_velocity = config.base_velocity * (1 - config.staking_pct)
        velocity[m] = eff_velocity

        # Price model: combination of equation-of-exchange and momentum
        annual_vol = vol * 12
        implied_price = annual_vol / (circ_supply[m] * eff_velocity) if circ_supply[m] > 0 and eff_velocity > 0 else current_price
        # Blend implied price with momentum (smoothing)
        current_price = 0.7 * implied_price + 0.3 * current_price
        current_price = max(current_price, 0.001)
        price[m] = current_price
        mcap[m] = current_price * circ_supply[m]

    return LaunchSimResult(
        months=months, circulating_supply=circ_supply,
        total_supply=total_sup, price=price, market_cap=mcap,
        volume=volume, fees_collected=fees, tokens_burned=burned,
        treasury_value=treasury, velocity=velocity
    )


print('Token launch simulator defined. Ready to configure and run.')

In [ ]:
# Configure and run the simulation
np.random.seed(42)

launch_config = TokenLaunchConfig(
    name='PROTO Token',
    total_supply=1_000_000_000,
    initial_price=0.10,
    allocations=allocations,  # reuse from Part 2
    monthly_volume_base=50_000_000,    # $50M/month initial
    monthly_volume_growth=0.05,        # 5% monthly growth
    fee_rate=0.003,                    # 0.3% fee rate
    burn_pct=0.50,                     # 50% of fees burned
    staking_pct=0.35,                  # 35% staked
    base_velocity=8.0                  # 8x base velocity
)

sim = simulate_token_launch(launch_config, duration_months=24)

print(f'Token Launch Simulation: {launch_config.name}')
print('=' * 60)
print(f'  Duration: 24 months')
print(f'  Initial price: ${launch_config.initial_price:.2f}')
print(f'  Final price:   ${sim.price[-1]:.4f}')
print(f'  Initial market cap: ${sim.market_cap[0]/1e6:,.1f}M')
print(f'  Final market cap:   ${sim.market_cap[-1]/1e6:,.1f}M')
print(f'  Total fees collected: ${sim.fees_collected.sum()/1e6:,.1f}M')
print(f'  Total tokens burned: {sim.tokens_burned.sum()/1e6:,.2f}M')
print(f'  Treasury balance: ${sim.treasury_value[-1]/1e6:,.1f}M')
print(f'  Final circulating supply: {sim.circulating_supply[-1]/1e6:,.1f}M')
print(f'  Final total supply: {sim.total_supply[-1]/1e6:,.1f}M')

In [ ]:
# Comprehensive visualization of the token launch
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Price
axes[0, 0].plot(sim.months, sim.price, color='#2ECC71', linewidth=2)
axes[0, 0].set_title('Token Price', fontsize=12)
axes[0, 0].set_xlabel('Month')
axes[0, 0].set_ylabel('Price ($)')
axes[0, 0].grid(True, alpha=0.3)

# 2. Market Cap
axes[0, 1].plot(sim.months, sim.market_cap / 1e6, color='#3498DB', linewidth=2)
axes[0, 1].set_title('Market Cap', fontsize=12)
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Market Cap ($M)')
axes[0, 1].grid(True, alpha=0.3)

# 3. Circulating vs Total Supply
axes[0, 2].plot(sim.months, sim.total_supply / 1e6, color='#E74C3C', linewidth=2, label='Total')
axes[0, 2].plot(sim.months, sim.circulating_supply / 1e6, color='#F39C12', linewidth=2, label='Circulating')
axes[0, 2].set_title('Supply Dynamics', fontsize=12)
axes[0, 2].set_xlabel('Month')
axes[0, 2].set_ylabel('Tokens (millions)')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# 4. Monthly Volume
axes[1, 0].bar(sim.months, sim.volume / 1e6, color='#9B59B6', alpha=0.7)
axes[1, 0].set_title('Monthly Trading Volume', fontsize=12)
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Volume ($M)')
axes[1, 0].grid(True, alpha=0.3)

# 5. Cumulative Fees & Burns
axes[1, 1].plot(sim.months, np.cumsum(sim.fees_collected) / 1e6, color='#2ECC71', linewidth=2, label='Fees')
axes[1, 1].plot(sim.months, sim.treasury_value / 1e6, color='#F39C12', linewidth=2, label='Treasury')
axes[1, 1].set_title('Cumulative Fees & Treasury', fontsize=12)
axes[1, 1].set_xlabel('Month')
axes[1, 1].set_ylabel('Value ($M)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# 6. Tokens Burned
axes[1, 2].bar(sim.months, sim.tokens_burned / 1e6, color='#E74C3C', alpha=0.7)
axes[1, 2].set_title('Monthly Tokens Burned', fontsize=12)
axes[1, 2].set_xlabel('Month')
axes[1, 2].set_ylabel('Tokens Burned (millions)')
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle(f'{launch_config.name} -- 24-Month Launch Simulation', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Dashboard: complete token launch lifecycle with price, supply, volume, fees, and burns.')

### Sensitivity Analysis

Let us test how changes in key parameters affect the final token price.

In [ ]:
# Sensitivity: vary burn percentage and staking percentage
np.random.seed(42)

burn_pcts = [0.0, 0.25, 0.50, 0.75, 1.0]
staking_pcts = [0.10, 0.25, 0.40, 0.60, 0.80]

results_grid = np.zeros((len(burn_pcts), len(staking_pcts)))

for i, bp in enumerate(burn_pcts):
    for j, sp in enumerate(staking_pcts):
        np.random.seed(42)  # reset seed for fair comparison
        cfg = TokenLaunchConfig(
            name='Test', total_supply=1_000_000_000,
            initial_price=0.10, allocations=allocations,
            monthly_volume_base=50_000_000, monthly_volume_growth=0.05,
            fee_rate=0.003, burn_pct=bp, staking_pct=sp, base_velocity=8.0
        )
        res = simulate_token_launch(cfg, 24)
        results_grid[i, j] = res.price[-1]

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(results_grid, cmap='YlGn', aspect='auto')
ax.set_xticks(range(len(staking_pcts)))
ax.set_xticklabels([f'{s:.0%}' for s in staking_pcts])
ax.set_yticks(range(len(burn_pcts)))
ax.set_yticklabels([f'{b:.0%}' for b in burn_pcts])
ax.set_xlabel('Staking %', fontsize=12)
ax.set_ylabel('Burn %', fontsize=12)
ax.set_title('Final Token Price -- Sensitivity to Burn % and Staking %', fontsize=13)

for i in range(len(burn_pcts)):
    for j in range(len(staking_pcts)):
        ax.text(j, i, f'${results_grid[i, j]:.3f}', ha='center', va='center', fontsize=10, fontweight='bold')

plt.colorbar(im, label='Token Price ($)')
plt.tight_layout()
plt.show()
print('Higher staking and higher burn both increase token price, but staking has a stronger effect.')

---

# Part 8: Exercises

Apply what you have learned to the following exercises. Each builds on the models defined above.

## Exercise 1: Custom Supply Model

Design a **hybrid supply model** that combines inflation and burning:
- Start with 500M tokens
- Year 1-3: 5% annual inflation (bootstrapping phase)
- Year 4+: 2% annual inflation + 3% annual burn
- Plot the supply curve over 20 years

**Hint:** Use a conditional inside your loop that checks the current year to switch between phases.

In [ ]:
# Exercise 1: Custom Hybrid Supply Model
# YOUR CODE HERE

def hybrid_supply_model(years: int = 20) -> Tuple[np.ndarray, np.ndarray]:
    """Implement a hybrid supply model with phased inflation and burn.

    Args:
        years: Number of years to simulate.

    Returns:
        Tuple of (time_years, supply_over_time).
    """
    # YOUR CODE HERE
    pass


## Exercise 2: Bonding Curve with Reserve Ratio

Implement a **Bancor-style** bonding curve where the reserve ratio determines the curve shape:

$$P = \frac{R}{S \times F}$$

Where R = reserve balance, S = token supply, F = reserve ratio (0 < F <= 1).

- Implement buy and sell functions
- Simulate 20 buys and 10 sells
- Test with reserve ratios of 0.1, 0.5, and 1.0
- Plot price trajectories for each ratio

**Hint:** When F = 1.0, price is constant (1:1 backed). When F < 1, price increases with supply.

In [ ]:
# Exercise 2: Bancor-Style Bonding Curve
# YOUR CODE HERE

def bancor_price(reserve: float, supply: float, reserve_ratio: float) -> float:
    """Calculate token price using the Bancor formula.

    Args:
        reserve: Current reserve balance.
        supply: Current token supply.
        reserve_ratio: Fractional reserve ratio (0 < F <= 1).

    Returns:
        Current token price.
    """
    # YOUR CODE HERE
    pass


## Exercise 3: veToken Fee Distribution

Build on the VeTokenLock class to simulate fee distribution over 52 weeks:
- Create 500 random lockers with varying amounts and durations
- Each week, $100,000 in fees is distributed proportional to voting power
- Track cumulative rewards for each lock duration bucket (1yr, 2yr, 3yr, 4yr)
- Plot cumulative rewards per bucket over time
- Calculate the annualized yield for each lock duration

**Hint:** At each week, compute each locker's share = their_power / total_power * weekly_fees.

In [ ]:
# Exercise 3: veToken Fee Distribution Simulation
# YOUR CODE HERE

def simulate_ve_fee_distribution(
    n_lockers: int = 500,
    weeks: int = 52,
    weekly_fees: float = 100_000.0
) -> Dict[str, np.ndarray]:
    """Simulate weekly fee distribution among veToken holders.

    Args:
        n_lockers: Number of token lockers.
        weeks: Number of weeks to simulate.
        weekly_fees: Protocol fees distributed each week in USD.

    Returns:
        Dictionary mapping lock duration labels to cumulative reward arrays.
    """
    # YOUR CODE HERE
    pass


## Exercise 4: Token Launch Parameter Sweep

Using the `simulate_token_launch` function, run a parameter sweep:
- Vary `fee_rate` from 0.1% to 1.0% (5 values)
- Vary `monthly_volume_growth` from 0% to 10% (5 values)
- For each combination, record the final market cap at month 24
- Create a heatmap showing market cap as a function of both parameters
- Identify which parameter has a larger impact on final market cap

**Hint:** Use nested loops similar to the sensitivity analysis in Part 7.

In [ ]:
# Exercise 4: Token Launch Parameter Sweep
# YOUR CODE HERE


---

# Summary

## What You Learned

- [x] **Token Supply Models**: Implemented fixed (Bitcoin), inflationary (Ethereum), and deflationary (burn) supply schedules and visualized their long-term trajectories
- [x] **Token Distribution & Vesting**: Modeled ICO/IDO allocations with cliff and linear vesting, visualized unlock timelines, and analyzed sell pressure from token unlocks
- [x] **Equation of Exchange (MV=PQ)**: Applied monetary economics to crypto tokens, showing how velocity affects implied token value and how staking acts as a velocity sink
- [x] **Bonding Curves**: Built linear, polynomial, and sigmoid bonding curves, simulated buy/sell dynamics with reserve tracking, and explored continuous fundraising mechanics
- [x] **Fee Distribution & Value Accrual**: Compared fee-burn (EIP-1559) and fee-distribution (staking rewards) models, analyzing how each channel accrues value to holders
- [x] **Governance Tokens (veToken Model)**: Simulated vote-escrowed mechanics with time-weighted voting power and governance vote outcomes
- [x] **Token Launch Simulation**: Combined all building blocks into an end-to-end simulator tracking price, supply, volume, fees, burns, and treasury over 24 months

## Key Insights

1. Supply model choice has profound effects on long-term holder incentives and price dynamics
2. Vesting schedules are critical for managing sell pressure -- cliff unlocks create sharp risk events
3. Token velocity is the most underappreciated factor in token valuation; staking and lockups are powerful velocity sinks
4. Bonding curves provide deterministic pricing and guaranteed liquidity without external exchanges
5. Fee-burn and fee-distribution are complementary strategies with different tax and behavioral implications
6. veToken mechanics align governance power with long-term commitment

## Next Steps

Continue to [Notebook 12: Governance Simulation](./12-governance-simulation.ipynb) to explore on-chain governance mechanisms, voting systems, proposal dynamics, and DAO treasury management in depth.

---